# LAB 1 – Bài 2 (Trang 30): Tiền xử lý dữ liệu môi trường

**Sinh viên:** Lê Nguyễn Anh Dự – **MSSV:** 6451071011

Thực hiện các yêu cầu a–d với dữ liệu từ Google Sheets. Các cột được gán theo thứ tự: `id`, `device_id`, `co_level`, `humidity`, `temperature`, `time`.

**Lưu ý quan trọng:** tệp `Dataset/tb_tracking.xlsx` trong thư mục bài tập có mốc thời gian từ 20/12/2016 đến 15/03/2017. Theo đúng mốc đề bài 03/01/2018, dữ liệu sau khi lọc sẽ rỗng. Notebook giữ nguyên mốc ngày được yêu cầu và sẽ báo rõ kết quả; cần kiểm tra lại năm nếu đề dự kiến tạo ra các biểu đồ có điểm dữ liệu.


In [ ]:
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

STUDENT_INFO = "Th?ng tin sinh vi?n: L? Nguy?n Anh D? - 6451071011"
SHEET_ID = "1Xizh3zvtMF4IdRNlueb74meOXk80-CHc"
XLSX_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=xlsx"
COLUMNS = ["id", "device_id", "co_level", "humidity", "temperature", "time"]

print(f"Downloading the Google Sheets workbook | {STUDENT_INFO}")
response = requests.get(XLSX_URL, timeout=60)
response.raise_for_status()
if not response.content.startswith(b"PK"):
    raise ValueError("Google Sheets did not return an XLSX file. Check the sheet sharing access.")
raw = pd.read_excel(io.BytesIO(response.content), header=None)
# Remove the first row only when the sheet itself includes column names.
if str(raw.iloc[0, 0]).strip().lower() == "id":
    raw = raw.iloc[1:].reset_index(drop=True)
if raw.shape[1] != len(COLUMNS):
    raise ValueError(f"Expected 6 columns, got {raw.shape[1]}. Check the selected worksheet.")
raw.columns = COLUMNS
raw["time"] = pd.to_datetime(raw["time"], errors="coerce", dayfirst=True)
df = raw
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns | {STUDENT_INFO}")


## Câu 1. Nhập dữ liệu và thống kê thông tin cơ bản

Gán tên sáu thuộc tính theo thứ tự đề bài, sau đó xem kiểu dữ liệu, giá trị thiếu và thống kê mô tả.


In [ ]:
print(f"Kích thước dữ liệu: {df.shape} | {STUDENT_INFO}")
print(f"Thông tin cột và kiểu dữ liệu | {STUDENT_INFO}"); df.info()
print(f"Số lượng giá trị thiếu | {STUDENT_INFO}"); display(df.isna().sum().to_frame("Số giá trị rỗng"))
print(f"Thống kê mô tả | {STUDENT_INFO}"); display(df.describe(include="all").T)
print(f"Khoảng thời gian trong dữ liệu nguồn: {df['time'].min()} đến {df['time'].max()} | {STUDENT_INFO}")


## Câu 2. Loại bỏ dữ liệu trước ngày 03/01/2018

Chuyển `time` sang datetime, loại dòng không có thời gian và giữ các phép đo từ 03/01/2018 trở đi.


In [ ]:
START_DATE = pd.Timestamp("2018-01-03")
before = len(df)
valid_time = df.dropna(subset=["time"]).copy()
df_clean = valid_time.loc[valid_time["time"] >= START_DATE].copy()
print(f"Dòng ban đầu: {before} | Dòng thiếu/không đọc được time đã loại: {before-len(valid_time)} | {STUDENT_INFO}")
print(f"Mốc lọc: {START_DATE} | Dòng trước mốc đã loại: {len(valid_time)-len(df_clean)} | Còn lại: {len(df_clean)} | {STUDENT_INFO}")
if df_clean.empty:
    print(f"CẢNH BÁO: Không còn dòng nào từ mốc yêu cầu. Hãy kiểm tra năm 2018 vì dữ liệu nguồn có thể kết thúc trước mốc này. | {STUDENT_INFO}")
else:
    print(f"Mẫu dữ liệu sau lọc | {STUDENT_INFO}"); display(df_clean.head())


## Câu 3. Tạo thuộc tính `minutes`

Tách giờ và phút từ `time`, tính số phút kể từ 00:00 theo công thức `hour × 60 + minute`. Giây không được tính vào đơn vị phút.


In [ ]:
df_clean["minutes"] = df_clean["time"].dt.hour * 60 + df_clean["time"].dt.minute
print(f"Dữ liệu sau khi tạo minutes | {STUDENT_INFO}")
display(df_clean[["id","device_id","time","minutes"]].head())
if not df_clean.empty:
    print(f"Khoảng minutes: {df_clean['minutes'].min()}–{df_clean['minutes'].max()} | {STUDENT_INFO}")


## Câu 4. Scatter plots theo `minutes`

Vẽ ba biểu đồ phân tán: `co_level`, `humidity` và `temperature` theo số phút trong ngày. Nếu bộ lọc ngày không giữ dòng nào, biểu đồ sẽ thông báo điều đó.


In [ ]:
plot_specs = [
    ("co_level", "Nồng độ khí CO", "CO"),
    ("humidity", "Độ ẩm", "Humidity"),
    ("temperature", "Nhiệt độ", "Temperature"),
]
for column, label, color in plot_specs:
    fig, ax = plt.subplots(figsize=(10, 5))
    if df_clean.empty:
        ax.text(0.5, 0.5, "Không có dữ liệu sau lọc ngày 03/01/2018", ha="center", va="center", transform=ax.transAxes)
    else:
        ax.scatter(df_clean["minutes"], df_clean[column], alpha=0.6, s=14, color=color)
    ax.set(title=f"{label} theo thời điểm trong ngày", xlabel="Thời điểm đo (phút từ 00:00)", ylabel=label)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    print(f"Scatter plot {column} theo minutes | {STUDENT_INFO}")
    plt.show()


## Kết luận

Các cột đã được đặt tên, thời gian được chuyển sang kiểu datetime, dữ liệu trước ngày yêu cầu được loại bỏ và `minutes` được tính từ giờ/phút. Hãy lưu ý cảnh báo về khoảng thời gian thực tế trong tệp nguồn: nếu mốc 2018 là lỗi đề bài và cần dùng năm khác, sửa `START_DATE` ở Câu 2 rồi chạy lại các ô phía dưới.
